In [33]:
# ─────────────────────────────────────────────────────────────────────
# CELL 0 – Environment Setup
# ─────────────────────────────────────────────────────────────────────
# load_dotenv() reads key=value pairs from a .env file and injects them
# into the process environment (os.environ).  We use an explicit path so
# the file is found regardless of what directory Jupyter is running from.
#
# The OPENAI_API_KEY variable is picked up automatically by OpenAI and
# LangChain clients – no need to pass it manually to every constructor.
# ─────────────────────────────────────────────────────────────────────

from dotenv import load_dotenv
import os

# Absolute path to the .env file that stores the OpenAI API key
env_path = r'C:\Users\user\OneDrive\Documents\Generative_AI\Langchain\VectorStrore\.env'

# Load variables from .env into os.environ; override=True ensures any
# previously set value is replaced with the one from the file
load_dotenv(dotenv_path=env_path, override=True)

# Quick sanity-check: print the first 20 characters of the key so we can
# confirm it loaded without exposing the full secret
key = os.getenv('OPENAI_API_KEY', '')
print(f'API key loaded: {key[:20]}...' if key else 'ERROR: API key not found!')


API key loaded: sk-proj-U4BG_7ReBqIY...


In [34]:
# ─────────────────────────────────────────────────────────────────────
# CELL 1 – Install Dependencies
# ─────────────────────────────────────────────────────────────────────
# Install all required packages in one shot.  Run this cell only once
# (or when setting up a fresh environment).
#
#  langchain           – Core LangChain framework (chains, prompts, etc.)
#  chromadb            – Chroma vector database backend
#  openai              – Official OpenAI Python SDK
#  tiktoken            – OpenAI tokenizer (used internally by LangChain)
#  pypdf               – PDF loader for ingesting PDF documents
#  langchain_openai    – LangChain integration for OpenAI models & embeddings
#  langchain-community – Community integrations (loaders, tools, etc.)
#  langchain-chroma    – Official LangChain ↔ Chroma integration package
#  python-dotenv       – Loads environment variables from a .env file
# ─────────────────────────────────────────────────────────────────────

!pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community langchain-chroma python-dotenv



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
# ─────────────────────────────────────────────────────────────────────
# CELL 2 – Import Vector Store & Embeddings
# ─────────────────────────────────────────────────────────────────────
# OpenAIEmbeddings  – Converts text into numerical vectors (embeddings)
#                     using OpenAI's text-embedding models (e.g.
#                     text-embedding-ada-002).  These vectors capture
#                     semantic meaning so similar texts are close together
#                     in vector space.
#
# Chroma            – A lightweight, open-source vector database that
#                     stores embeddings and supports fast similarity search.
#                     Imported from langchain_chroma (the dedicated
#                     integration package for LangChain >= 1.x).
# ─────────────────────────────────────────────────────────────────────

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma


In [36]:
# ─────────────────────────────────────────────────────────────────────
# CELL 3 – Create LangChain Documents
# ─────────────────────────────────────────────────────────────────────
# A LangChain Document is the standard data unit for text storage and
# retrieval.  Each Document has two fields:
#
#   page_content  – The raw text that will be embedded and searched.
#   metadata      – A dictionary of arbitrary key-value pairs attached to
#                   the document.  Metadata is stored alongside the vector
#                   and can be used for filtering at query time.
#
# Here we create 5 documents about famous IPL cricketers, each tagged
# with the player's team so we can filter by team later.
# ─────────────────────────────────────────────────────────────────────

from langchain_core.documents import Document

# Document 1 – Virat Kohli (Royal Challengers Bangalore)
doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )

# Document 2 – Rohit Sharma (Mumbai Indians)
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )

# Document 3 – MS Dhoni (Chennai Super Kings)
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )

# Document 4 – Jasprit Bumrah (Mumbai Indians)
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )

# Document 5 – Ravindra Jadeja (Chennai Super Kings)
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [37]:
# ─────────────────────────────────────────────────────────────────────
# CELL 4 – Bundle Documents into a List
# ─────────────────────────────────────────────────────────────────────
# Combine all individual Document objects into a single list.
# This list is passed to the vector store's add_documents() method
# which processes them in batch – more efficient than adding one by one.
# ─────────────────────────────────────────────────────────────────────

docs = [doc1, doc2, doc3, doc4, doc5]
print(f'{len(docs)} documents ready to be indexed.')


5 documents ready to be indexed.


In [38]:
# ─────────────────────────────────────────────────────────────────────
# CELL 5 – Initialize the Chroma Vector Store
# ─────────────────────────────────────────────────────────────────────
# Create (or reopen) a Chroma vector store with the following config:
#
#   embedding_function  – The model used to convert text → vectors.
#                         OpenAIEmbeddings() reads OPENAI_API_KEY from
#                         the environment automatically.
#
#   persist_directory   – Local folder where Chroma saves its data.
#                         If the folder already exists, the store is
#                         reopened; otherwise a new one is created.
#
#   collection_name     – Logical namespace inside the database.
#                         Multiple collections can coexist in the same
#                         persist_directory.
# ─────────────────────────────────────────────────────────────────────

vector_store = Chroma(
    embedding_function=OpenAIEmbeddings(),  # Uses text-embedding-ada-002 by default
    persist_directory='my_chroma_db',       # Data saved to ./my_chroma_db/
    collection_name='sample'                # Collection namespace
)
print('Chroma vector store initialised successfully.')


Chroma vector store initialised successfully.


In [39]:
# ─────────────────────────────────────────────────────────────────────
# CELL 6 – Add Documents to the Vector Store
# ─────────────────────────────────────────────────────────────────────
# add_documents() does three things for each Document:
#   1. Extracts the page_content text.
#   2. Calls the embedding_function to convert text → a float vector.
#   3. Stores the vector + metadata + original text in the Chroma DB.
#
# The method returns a list of UUIDs – one per document – which can be
# used later to update or delete specific entries.
# ─────────────────────────────────────────────────────────────────────

# add documents
added_ids = vector_store.add_documents(docs)
print(f'Added {len(added_ids)} documents. IDs: {added_ids}')


Added 5 documents. IDs: ['0d08c6bb-a8bb-45dd-b1b0-175a00f571eb', 'cdca4700-04b4-44d8-bfca-c770d984924b', '4473fccd-115f-40ae-8b1f-b3063163273b', 'a5b32f5b-09af-443c-8869-67e337625060', 'f2253ca9-a30a-49ba-9e6a-aec091e66f0f']


In [40]:
# ─────────────────────────────────────────────────────────────────────
# CELL 7 – View All Documents in the Store
# ─────────────────────────────────────────────────────────────────────
# vector_store.get() retrieves all stored entries.
# The 'include' list controls which fields are returned:
#
#   'embeddings'  – The raw float vectors (useful for inspection/debugging).
#   'documents'   – The original page_content text strings.
#   'metadatas'   – The metadata dicts attached to each document.
#
# The returned dict also always includes 'ids' – the UUID of every entry.
# ─────────────────────────────────────────────────────────────────────

# view all documents with their embeddings and metadata
vector_store.get(include=['embeddings', 'documents', 'metadatas'])


{'ids': ['6fa9ffd0-b1b5-4075-bbd9-bf3f9a19df00',
  '7afaca7e-7bb1-481d-a95e-2e0eeb165e4d',
  '2360d0dc-20e9-47d2-800c-088b5295ef7c',
  '4e061b4b-12d2-4bf9-ad00-d9084bc87928',
  '0d08c6bb-a8bb-45dd-b1b0-175a00f571eb',
  'cdca4700-04b4-44d8-bfca-c770d984924b',
  '4473fccd-115f-40ae-8b1f-b3063163273b',
  'a5b32f5b-09af-443c-8869-67e337625060',
  'f2253ca9-a30a-49ba-9e6a-aec091e66f0f'],
 'embeddings': array([[-0.00268021, -0.00010323,  0.02815653, ..., -0.01501936,
          0.00590092, -0.01164922],
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        [-0.02720404,  0.00912717,  0.02682374, ..., -0.02629133,
          0.00909548, -0.01973751],
        ...,
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        [-0.02714536,  0.00885395,  0.02699314, ..., -0.02592762,
          0.00900617, -0.01999116],
        [-0.01810451,  0.01281202,  0.0347942 , ..., -0.03034012,
         -0

In [41]:
# ─────────────────────────────────────────────────────────────────────
# CELL 8 – Semantic Similarity Search
# ─────────────────────────────────────────────────────────────────────
# similarity_search() finds the k most semantically similar documents
# to the query string.  It works by:
#   1. Embedding the query using the same embedding_function.
#   2. Computing cosine similarity between the query vector and all
#      stored vectors.
#   3. Returning the top-k closest Document objects.
#
# Unlike keyword search, this understands meaning – so 'bowler' will
# match documents that mention bowling-related skills even if the exact
# word doesn't appear in the document.
# ─────────────────────────────────────────────────────────────────────

# search documents – returns top-2 most relevant Document objects
results = vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2  # Return the 2 most similar documents
)
results


[Document(id='2360d0dc-20e9-47d2-800c-088b5295ef7c', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='a5b32f5b-09af-443c-8869-67e337625060', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [42]:
# ─────────────────────────────────────────────────────────────────────
# CELL 9 – Similarity Search WITH Score
# ─────────────────────────────────────────────────────────────────────
# similarity_search_with_score() is the same as similarity_search() but
# also returns a relevance/distance score for each result.
#
# Chroma uses L2 (Euclidean) distance by default:
#   - Score = 0.0  → perfect match (identical vectors)
#   - Lower score  → more similar  (shorter distance)
#   - Higher score → less similar  (farther distance)
#
# Returns a list of (Document, float) tuples.
# ─────────────────────────────────────────────────────────────────────

# search with similarity score – returns (Document, distance_score) tuples
results_with_scores = vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2  # Return the 2 most similar documents
)
results_with_scores


[(Document(id='2360d0dc-20e9-47d2-800c-088b5295ef7c', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.35404834151268005),
 (Document(id='a5b32f5b-09af-443c-8869-67e337625060', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.35445845127105713)]

In [43]:
# ─────────────────────────────────────────────────────────────────────
# CELL 10 – Metadata Filtering
# ─────────────────────────────────────────────────────────────────────
# The 'filter' parameter restricts the search to only documents whose
# metadata matches the given condition BEFORE the vector similarity
# comparison is done.  This is a pre-filter (not post-filter).
#
# Here we search with an empty query string and filter by team, which
# effectively retrieves ALL documents belonging to 'Chennai Super Kings'
# ranked by their distance to an empty embedding.
#
# Practical use-case: "Give me all CSK players sorted by relevance
# to a specific topic" – just replace query="" with a real question.
# ─────────────────────────────────────────────────────────────────────

# meta-data filtering – only return documents where team == 'Chennai Super Kings'
filtered_results = vector_store.similarity_search_with_score(
    query="",                                      # Empty query → no semantic bias
    filter={"team": "Chennai Super Kings"}         # Only CSK players
)
filtered_results


[(Document(id='7afaca7e-7bb1-481d-a95e-2e0eeb165e4d', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.6488258242607117),
 (Document(id='4473fccd-115f-40ae-8b1f-b3063163273b', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.6488258242607117),
 (Document(id='f2253ca9-a30a-49ba-9e6a-aec091e66f0f', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.6566494703292847),
 (Document(id='4e061b4b-12d2-4bf9-ad00-d9084bc87928', metadata={'team': 'Chennai S

In [44]:
# ─────────────────────────────────────────────────────────────────────
# CELL 11 – Update a Document
# ─────────────────────────────────────────────────────────────────────
# update_document() replaces an existing document's content and re-
# computes its embedding so the vector index stays accurate.
#
# Why update?  The original doc1 had a short description of Kohli.
# Here we replace it with a more detailed, richer description so that
# future similarity searches return better-quality results.
#
# We fetch the IDs dynamically (instead of hardcoding a UUID) so this
# cell works correctly on every fresh run of the notebook.
# ─────────────────────────────────────────────────────────────────────

# Create the updated version of doc1 (Virat Kohli) with richer content
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

# Fetch all current IDs dynamically – the first ID corresponds to doc1
doc_ids = vector_store.get()["ids"]
if doc_ids:
    # Replace the document at doc_ids[0] with updated_doc1
    vector_store.update_document(document_id=doc_ids[0], document=updated_doc1)
    print(f"Updated document ID: {doc_ids[0]}")
else:
    print("No documents found to update.")


Updated document ID: 6fa9ffd0-b1b5-4075-bbd9-bf3f9a19df00


In [45]:
# ─────────────────────────────────────────────────────────────────────
# CELL 12 – Verify the Update
# ─────────────────────────────────────────────────────────────────────
# Retrieve all documents again to confirm that doc1 now contains the
# updated (longer) description for Virat Kohli and that its embedding
# vector has been recomputed accordingly.
#
# Compare the 'documents' field here with Cell 7 output – you should
# see the longer Kohli text at position 0.
# ─────────────────────────────────────────────────────────────────────

# view documents after update – confirm doc1 content has changed
vector_store.get(include=['embeddings', 'documents', 'metadatas'])


{'ids': ['6fa9ffd0-b1b5-4075-bbd9-bf3f9a19df00',
  '7afaca7e-7bb1-481d-a95e-2e0eeb165e4d',
  '2360d0dc-20e9-47d2-800c-088b5295ef7c',
  '4e061b4b-12d2-4bf9-ad00-d9084bc87928',
  '0d08c6bb-a8bb-45dd-b1b0-175a00f571eb',
  'cdca4700-04b4-44d8-bfca-c770d984924b',
  '4473fccd-115f-40ae-8b1f-b3063163273b',
  'a5b32f5b-09af-443c-8869-67e337625060',
  'f2253ca9-a30a-49ba-9e6a-aec091e66f0f'],
 'embeddings': array([[-0.00544442, -0.01907989,  0.00706373, ..., -0.01627786,
         -0.00032134,  0.00724619],
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        [-0.02720404,  0.00912717,  0.02682374, ..., -0.02629133,
          0.00909548, -0.01973751],
        ...,
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        [-0.02714536,  0.00885395,  0.02699314, ..., -0.02592762,
          0.00900617, -0.01999116],
        [-0.01810451,  0.01281202,  0.0347942 , ..., -0.03034012,
         -0

In [46]:
# ─────────────────────────────────────────────────────────────────────
# CELL 13 – Delete a Document
# ─────────────────────────────────────────────────────────────────────
# delete() removes one or more documents from the vector store by ID.
# Both the stored vector and the original text/metadata are permanently
# removed from the Chroma collection.
#
# Again we use a dynamic ID lookup so the cell works on any run.
# ─────────────────────────────────────────────────────────────────────

# delete the first document from the store
doc_ids = vector_store.get()["ids"]
if doc_ids:
    vector_store.delete(ids=[doc_ids[0]])  # Pass a list of IDs to delete
    print(f"Deleted document ID: {doc_ids[0]}")
else:
    print("No documents found to delete.")


Deleted document ID: 6fa9ffd0-b1b5-4075-bbd9-bf3f9a19df00


In [47]:
# ─────────────────────────────────────────────────────────────────────
# CELL 14 – Verify the Deletion
# ─────────────────────────────────────────────────────────────────────
# Retrieve all documents one final time.  The store should now contain
# only 4 documents (doc2–doc5) since we deleted the first one in
# Cell 13.  Confirm by checking the 'ids' and 'documents' fields.
# ─────────────────────────────────────────────────────────────────────

# view documents after deletion – should show 4 remaining documents
vector_store.get(include=['embeddings', 'documents', 'metadatas'])


{'ids': ['7afaca7e-7bb1-481d-a95e-2e0eeb165e4d',
  '2360d0dc-20e9-47d2-800c-088b5295ef7c',
  '4e061b4b-12d2-4bf9-ad00-d9084bc87928',
  '0d08c6bb-a8bb-45dd-b1b0-175a00f571eb',
  'cdca4700-04b4-44d8-bfca-c770d984924b',
  '4473fccd-115f-40ae-8b1f-b3063163273b',
  'a5b32f5b-09af-443c-8869-67e337625060',
  'f2253ca9-a30a-49ba-9e6a-aec091e66f0f'],
 'embeddings': array([[ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        [-0.02720404,  0.00912717,  0.02682374, ..., -0.02629133,
          0.00909548, -0.01973751],
        [-0.0168051 ,  0.01254483,  0.03444914, ..., -0.02975629,
         -0.00547608,  0.00534827],
        ...,
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        [-0.02714536,  0.00885395,  0.02699314, ..., -0.02592762,
          0.00900617, -0.01999116],
        [-0.01810451,  0.01281202,  0.0347942 , ..., -0.03034012,
         -0.00595078,  0.00521716]], shape=(8, 1536))